In [0]:
from pyspark.sql.functions import col, count, sum, when, lit

silver_df = spark.table("workspace.default.silver_transactions")

total_rows = silver_df.count()

null_amount = silver_df.filter(col("Amount").isNull()).count()

invalid_class = silver_df.filter(
    ~col("Class").isin(0, 1)
).count()

negative_amount = silver_df.filter(
    col("Amount") < 0
).count()

duplicate_rows = (
    silver_df
    .groupBy(silver_df.columns)
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Total rows:", total_rows)
print("Null Amount:", null_amount)
print("Invalid Class:", invalid_class)
print("Negative Amount:", negative_amount)
print("Duplicate groups:", duplicate_rows)

Total rows: 283726
Null Amount: 0
Invalid Class: 0
Negative Amount: 0
Duplicate groups: 0


In [0]:
from pyspark.sql import Row
from datetime import datetime

dq_results = [
    Row(
        check_name="null_amount",
        check_description="Amount should not be null",
        failed_records=null_amount,
        status="PASS" if null_amount == 0 else "FAIL"
    ),
    Row(
        check_name="invalid_class",
        check_description="Class must be 0 or 1",
        failed_records=invalid_class,
        status="PASS" if invalid_class == 0 else "FAIL"
    ),
    Row(
        check_name="negative_amount",
        check_description="Amount should not be negative",
        failed_records=negative_amount,
        status="PASS" if negative_amount == 0 else "FAIL"
    ),
    Row(
        check_name="duplicate_groups",
        check_description="Duplicate records should not exist in Silver",
        failed_records=duplicate_rows,
        status="PASS" if duplicate_rows == 0 else "FAIL"
    )
]

dq_df = spark.createDataFrame(dq_results)

dq_df = dq_df.withColumn(
    "dq_run_time",
    lit(datetime.now())
)

display(dq_df)

check_name,check_description,failed_records,status,dq_run_time
null_amount,Amount should not be null,0,PASS,2026-09-26T09:13:15.834Z
invalid_class,Class must be 0 or 1,0,PASS,2026-09-26T09:13:15.834Z
negative_amount,Amount should not be negative,0,PASS,2026-09-26T09:13:15.834Z
duplicate_groups,Duplicate records should not exist in Silver,0,PASS,2026-09-26T09:13:15.834Z


In [0]:
(
    dq_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.data_quality_results")
)

print("Data quality results table created successfully!")

Data quality results table created successfully!


In [0]:
%sql
SELECT *
FROM workspace.default.data_quality_results;

check_name,check_description,failed_records,status,dq_run_time
null_amount,Amount should not be null,0,PASS,2026-09-26T09:13:15.834Z
invalid_class,Class must be 0 or 1,0,PASS,2026-09-26T09:13:15.834Z
negative_amount,Amount should not be negative,0,PASS,2026-09-26T09:13:15.834Z
duplicate_groups,Duplicate records should not exist in Silver,0,PASS,2026-09-26T09:13:15.834Z
